In [0]:
pip install openpyxl

pip install xlrd

In [0]:
%restart_python

Environment Variables and imports

In [0]:
import pandas as pd
from pyspark.sql import functions as F

vol="/Volumes/wa_fiscal/bronze/raw/"


## Inspecting the data before we parse it for ingestion

In [0]:


peek = pd.read_excel(
    f"{vol}/CMO-Historical-Data-Monthly.xlsx",
    sheet_name="Monthly Prices",
    header=None,
    nrows=12,
)
display(peek)

## Iron ore data ingest to bronze layer

In [0]:
SKIP = 4 #Skip first 4 rows in excel

prices = pd.read_excel(
    f"{vol}/CMO-Historical-Data-Monthly.xlsx",
    sheet_name="Monthly Prices",
    skiprows=SKIP,
)

prices = prices.rename(columns={prices.columns[0]:"period"})
iron_col = [c for c in prices.columns if "iron" in str(c).lower()][0]

prices = prices[["period",iron_col]].rename(columns={iron_col:"price_usd_per_tonne"})

prices["price_usd_per_tonne"] = pd.to_numeric(prices["price_usd_per_tonne"],errors="coerce")
prices = prices.dropna(subset=["price_usd_per_tonne"])
prices["period"] = prices["period"].astype(str)

df=(
    spark.createDataFrame(prices)
    .withColumn("_ingested_at",F.current_timestamp())
    .withColumn("_source_file",F.lit("CMO-Historical-Data-Monthly.xlsx"))
)
(
    df
    .write    
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("wa_fiscal.bronze.commodity_prices"))

display(spark.table("wa_fiscal.bronze.commodity_prices").limit(10))


Check data in Table

In [0]:
%sql
select * from wa_fiscal.bronze.commodity_prices
limit 10

## Ingest exchange rate to Bronze layer

In [0]:
fx = pd.read_excel(f"{vol}/f11hist.xls",sheet_name="Data")
display(fx)

In [0]:
fx = pd.read_excel(f"{vol}/f11hist.xls",sheet_name="Data",skiprows=10)
fx = fx.rename(columns={fx.columns[0]:"period"})
display(fx)
usd_col = [c for c in fx.columns if "USD" in str(c).upper()][0]
inr_col = [c for c in fx.columns if "RIRE" in str(c).upper()][0]
fx=fx[["period",usd_col,inr_col]].rename(columns={usd_col:"aud_usd",inr_col:"aud_inr"})
fx = fx.dropna(subset=["aud_usd","aud_inr"])
fx["period"] = fx["period"].astype(str)

df_fx = (
    spark.createDataFrame(fx)
    .withColumn("_ingested_at",F.current_timestamp())
    .withColumn("_source_file",F.lit("f11hist.xls"))
)
(    
 df_fx.write
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("wa_fiscal.bronze.fx_rates")
)

Check data in FX table

In [0]:
%sql
select * from wa_fiscal.bronze.fx_rates limit 10

## Seed data to bronze

In [0]:
seed = (spark.read
        .option("header",True)
        .option("inferSchema",True)
        .csv(f"{vol}/treasury_assumptions.csv")
        .withColumn("_ingested_at",F.current_timestamp())
       )
(
    seed.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("wa_fiscal.bronze.treasury_assumptions")
)


Check data in Treasury_assumptions table

In [0]:
%sql
select * from wa_fiscal.bronze.treasury_assumptions